In [1]:
from matplotlib import pyplot as plt
import numpy as np

import torch
from sklearn.metrics import classification_report

from tqdm import tqdm

import time

from PieceDetection import PieceDetection

from Dataset.DataSetLoaders import ChessDataset

In [2]:
ds = ChessDataset.ChessDataset(
    config={
        "img_size": (640,640)
    }
)

In [3]:
pc_cnn = PieceDetection.PieceDetector("cnn")
pc_yolo = PieceDetection.PieceDetector("yolo")

RuntimeError: Error(s) in loading state_dict for PieceDetectorCNNModel:
	size mismatch for occupancy_classifier_heads.0.weight: copying a param with shape torch.Size([38, 512]) from checkpoint, the shape in current model is torch.Size([128, 512]).
	size mismatch for occupancy_classifier_heads.0.bias: copying a param with shape torch.Size([38]) from checkpoint, the shape in current model is torch.Size([128]).
	size mismatch for occupancy_classifier_heads.3.weight: copying a param with shape torch.Size([0, 38]) from checkpoint, the shape in current model is torch.Size([1, 128]).
	size mismatch for occupancy_classifier_heads.3.bias: copying a param with shape torch.Size([0]) from checkpoint, the shape in current model is torch.Size([1]).
	size mismatch for piece_color_classifier_heads.0.weight: copying a param with shape torch.Size([38, 512]) from checkpoint, the shape in current model is torch.Size([128, 512]).
	size mismatch for piece_color_classifier_heads.0.bias: copying a param with 

In [4]:
accs = []
avg_preprocess_time = 0
avg_process_time = 0
for img, label in tqdm(ds):
    try:
        pc_cnn.set_img(img, label["corners"])
        start_time = time.perf_counter()
        pc_cnn.preprocess()
        interm_time = time.perf_counter()
        preds = pc_cnn.predict()
        end_time = time.perf_counter()
        avg_preprocess_time += interm_time - start_time
        avg_process_time += end_time - interm_time

        acc = (preds.argmax(dim=1).squeeze() == label["board_tensor"]).sum().item()
        accs.append(acc)
    except:
        pass

avg_preprocess_time /= len(accs)
avg_process_time /= len(accs)
accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f} | Avg Pre-Processing Time: {avg_preprocess_time*1e3:.3f}ms | Avg Processing Time: {avg_process_time*1e3:.3f}ms")

100%|██████████| 1009/1009 [04:38<00:00,  3.62it/s]

Acc: 0.82 | Errors: 11.5610 | Avg Pre-Processing Time: 231.455ms | Avg Processing Time: 27.161ms


In [8]:
accs = []
avg_preprocess_time = 0
avg_process_time = 0
for img, label in tqdm(ds):
    try:
        pc_yolo.set_img(img, label["corners"])
        start_time = time.perf_counter()
        pc_yolo.preprocess()
        interm_time = time.perf_counter()
        preds = pc_yolo.predict()
        end_time = time.perf_counter()
        avg_preprocess_time += interm_time - start_time
        avg_process_time += end_time - interm_time
        
        acc = (preds.squeeze() == label["board_tensor"]).sum().item()
        accs.append(acc)
    except:
        pass


avg_preprocess_time /= len(accs)
avg_process_time /= len(accs)
accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f} | Avg Pre-Processing Time: {avg_preprocess_time*1e3:.3f}ms | Avg Processing Time: {avg_process_time*1e3:.3f}ms")

100%|██████████| 1009/1009 [00:40<00:00, 24.76it/s]

Acc: 0.90 | Errors: 6.4975 | Avg Pre-Processing Time: 0.145ms | Avg Processing Time: 14.438ms


In [ ]:
train_ds, valid_ds, test_ds = ChessDataset.ChessDataset.train_valid_test_split(ds, sizes=(.8,.1,.1), random_state=42)

In [ ]:
accs = []
actual = []
preds_all = []
avg_preprocess_time = 0
avg_process_time = 0

for img, label in tqdm(test_ds):
    try:
        pc_cnn.set_img(img, label["corners"])
        start_time = time.perf_counter()
        pc_cnn.preprocess()
        interm_time = time.perf_counter()
        preds = pc_cnn.predict()
        end_time = time.perf_counter()
        avg_preprocess_time += interm_time - start_time
        avg_process_time += end_time - interm_time

        acc = (preds.argmax(dim=1).squeeze() == label["board_tensor"]).sum().item()
        preds_all.append(preds.argmax(dim=1).reshape(-1))
        actual.append(label["board_tensor"].reshape(-1))
        accs.append(acc)
    except:
        pass

avg_preprocess_time /= len(accs)
avg_process_time /= len(accs)
accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f} | Avg Pre-Processing Time: {avg_preprocess_time*1e3:.3f}ms | Avg Processing Time: {avg_process_time*1e3:.3f}ms")

actual = torch.cat(actual, dim=0)
preds_all = torch.cat(preds_all, dim=0)
rep = classification_report(actual, preds_all, target_names=['White-Pawn', 'White-Knight', 'White-Bishop', 'White-Rook', 'White-Queen', 'White-King', 'Black-Pawn', 'Black-Knight', 'Black-Bishop', 'Black-Rook', 'Black-Queen', 'Black-King', 'Empty'])
print(rep)

100%|██████████| 103/103 [00:50<00:00,  2.06it/s]

Acc: 0.65 | Errors: 22.1845 | Avg Pre-Processing Time: 227.945ms | Avg Processing Time: 246.887ms
              precision    recall  f1-score   support

  White-Pawn       0.00      0.00      0.00       588
White-Knight       0.00      0.00      0.00       107
White-Bishop       0.00      0.00      0.00       107
  White-Rook       0.00      0.00      0.00       161
 White-Queen       0.00      0.00      0.00        75
  White-King       0.00      0.00      0.00       103
  Black-Pawn       0.00      0.00      0.00       603
Black-Knight       0.00      0.00      0.00       107
Black-Bishop       0.00      0.00      0.00        95
  Black-Rook       0.00      0.00      0.00       163
 Black-Queen       0.00      0.00      0.00        73
  Black-King       0.00      0.00      0.00       103
       Empty       0.65      1.00      0.79      4307

    accuracy                           0.65      6592
   macro avg       0.05      0.08      0.06      6592
weighted avg       0.43      0.65   


/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()

In [5]:
next(PieceDetection.PieceDetection_CNN.piece_detection_model.parameters()).device

device(type='cpu')

In [6]:
accs = []
actual = []
preds_all = []
for img, label in tqdm(test_ds):
    try:
        pc_yolo.set_img(img, label["corners"])
        pc_yolo.preprocess()
        preds = pc_yolo.predict()

        acc = (preds.squeeze() == label["board_tensor"]).sum().item()
        preds_all.append(preds.reshape(-1))
        actual.append(label["board_tensor"].reshape(-1))
        accs.append(acc)
    except:
        pass

accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f}")

actual = torch.cat(actual, dim=0)
preds_all = torch.cat(preds_all, dim=0)
rep = classification_report(actual, preds_all)
print(rep)

100%|██████████| 103/103 [00:04<00:00, 25.58it/s]

Acc: 0.91 | Errors: 5.8738
              precision    recall  f1-score   support

           0       0.97      0.68      0.80       588
           1       0.34      0.79      0.47       107
           2       0.73      0.59      0.65       107
           3       0.94      0.71      0.81       161
           4       0.88      0.60      0.71        75
           5       0.91      0.81      0.86       103
           6       0.97      0.75      0.85       603
           7       0.53      0.97      0.69       107
           8       0.65      0.85      0.74        95
           9       0.76      0.87      0.81       163
          10       0.89      0.75      0.81        73
          11       0.76      0.88      0.82       103
          12       0.96      0.99      0.98      4307

    accuracy                           0.91      6592
   macro avg       0.79      0.79      0.77      6592
weighted avg       0.93      0.91      0.91      6592

